In [1]:
import os
import requests
from dotenv import load_dotenv

load_dotenv()

SEMANTIC_SCHOLAR_API_KEY = os.getenv("SEMANTIC_SCHOLAR_API_KEY")
PDF_DIR = "../data/pdfs"

In [2]:
def search_papers(topic, limit=20, offset=0):
    url = "https://api.semanticscholar.org/graph/v1/paper/search"
    params = {
        "query": topic,
        "limit": limit,
        "offset": offset,
        "fields": "title,year,citationCount,openAccessPdf"
    }
    headers = {"x-api-key": SEMANTIC_SCHOLAR_API_KEY}

    response = requests.get(url, params=params, headers=headers)

    if response.status_code != 200:
        print("API error:", response.status_code)
        return []

    data = response.json()
    return data.get("data", [])


In [3]:
def rank_papers(papers, top_k=3):
    return sorted(
        papers,
        key=lambda p: (p.get("citationCount",0), p.get("year",0)),
        reverse=True
    )[:top_k]


In [4]:
def is_valid_pdf(content):
    return content[:4] == b"%PDF"


def download_until_n_pdfs(topic, required=3, batch_size=20):
    downloaded = 0
    offset = 0
    checked_titles = set()

    while downloaded < required:
        papers = search_papers(topic, limit=batch_size, offset=offset)

        if not papers:
            print("No more papers available.")
            break

        for p in papers:
            if downloaded >= required:
                break

            title = p.get("title", "Unknown")

            if title in checked_titles:
                continue
            checked_titles.add(title)

            pdf_info = p.get("openAccessPdf")
            if not pdf_info or not pdf_info.get("url"):
                continue

            pdf_url = pdf_info["url"]

            try:
                response = requests.get(
                    pdf_url,
                    timeout=20,
                    headers={"User-Agent": "Mozilla/5.0"}
                )

                if response.status_code != 200:
                    continue

                content = response.content

                # 🔑 CRITICAL CHECK
                if not is_valid_pdf(content):
                    print(f"Invalid PDF skipped → {title}")
                    continue

                downloaded += 1
                with open(f"{PDF_DIR}/paper_{downloaded}.pdf", "wb") as f:
                    f.write(content)

                print(f"Downloaded valid PDF → paper_{downloaded}.pdf")

            except Exception as e:
                print("Download error:", e)

        offset += batch_size

    print(f"\nTotal valid PDFs downloaded: {downloaded}")


In [6]:
topic = input("Enter research topic: ").strip()
print(f"Searching best papers related to {topic}")
download_until_n_pdfs(topic, required=3)


Searching best papers related to NLP
Downloaded valid PDF → paper_1.pdf
Downloaded valid PDF → paper_2.pdf
Downloaded valid PDF → paper_3.pdf

Total valid PDFs downloaded: 3
